In [1]:
from opt_targeted_transfers import BinaryGapTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
tt = BinaryGapTargetedTransfers(c_bar=2.15, n_transfer_values=5)

In [4]:
# Nuisance parameter estimation
# Fit conditional improvement regressors for different transfer values
tt.fit(train_dataset, validation_dataset)

Fitting conditional gap improvement for transfer size 0.01


  0%|          | 0/300 [00:00<?, ?it/s]

> /zfs/gsb/intermediate-yens/rsahoo/poverty/package/src/opt_targeted_transfers/conditional_improvement.py(154)get_conditional_improvement_regressor()
    152             import pdb
    153             pdb.set_trace()
--> 154             loss.backward()
    155             optimizer.step()
    156 

tensor([1.9726e-04, 2.3497e-04, 1.0871e-04, 1.1016e-04, 6.3543e-05, 3.8774e-04,
        8.5499e-05, 2.0971e-04, 4.4347e-05, 8.4274e-05, 2.5171e-04, 2.2991e-04,
        1.9877e-04, 2.9595e-04, 2.6850e-04, 2.3371e-04, 1.8121e-04, 1.3969e-04,
        7.6030e-05, 2.3037e-04, 3.2095e-04, 1.4508e-04, 1.0871e-04, 9.4596e-05,
        3.5217e-04, 1.2100e-04, 2.6513e-04, 2.0936e-04, 2.1175e-04, 4.5223e-05,
        2.1018e-04, 3.2329e-04, 5.2381e-04, 1.0399e-04, 2.5681e-04, 1.9864e-04,
        2.3466e-04, 6.3724e-04, 3.9665e-04, 8.0001e-05, 6.5130e-05, 3.8375e-05,
        2.3281e-04, 3.0957e-05, 3.8227e-04, 8.9334e-05, 2.5423e-04, 1.0494e-04,
        1.2252e-04, 1.2162e-04, 2.4604e-04, 2.3948e-04, 1.77

  0%|          | 1/300 [01:59<9:57:10, 119.84s/it, val loss=1.03]

> /zfs/gsb/intermediate-yens/rsahoo/poverty/package/src/opt_targeted_transfers/conditional_improvement.py(153)get_conditional_improvement_regressor()
    151             loss = torch.sum(unweighted_loss * weights)
    152             import pdb
--> 153             pdb.set_trace()
    154             loss.backward()
    155             optimizer.step()

tensor([1.6445, 1.6445, 0.5218, 1.6445, 0.5218, 0.5218, 0.5218, 1.6445, 0.5218,
        0.3233, 1.6445, 0.3233, 1.6445, 0.5218, 2.0626, 0.3233, 0.5218, 1.6445,
        0.3233, 0.5218, 0.3233, 1.6445, 0.5218, 1.6445, 1.6445, 0.5218, 1.6445,
        0.5218, 0.5218, 0.5218, 0.3233, 0.5218, 1.6445, 0.5218, 0.3233, 1.6445,
        0.5218, 1.6445, 1.6445, 0.5218, 0.5218, 1.6445, 0.5218, 0.3233, 0.5218,
        1.6445, 1.6445, 1.6445, 0.5218, 1.6445, 1.6445, 0.3233, 0.5218, 0.5218,
        1.6445, 1.6445, 0.3233, 0.3233, 0.5218, 1.6445, 0.5218, 0.5218, 0.3233,
        0.5218, 1.6445, 0.5218, 0.5218, 0.5218, 0.5218, 0.5218, 0.5218, 0.5218,
     

  0%|          | 1/300 [04:24<21:55:38, 264.01s/it, val loss=1.03]


In [5]:
# Precomputation for policy optimization step.
tt.optimize_transfers_for_budget_grid(test_covariate_dataset, budgets=[0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.15])

In [6]:
# Set budget and run policy optimization step for that budget.
# Policy optimization step returns transfer amount for each unit in the test set.
# Note that budget must lie in the set of budgets used in the precomputation step.
tt.set_budget(budget=1.0)
assignments = tt.run_opt(test_covariate_dataset)

In [7]:
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.04450745063123372,
 'post_transfer_poverty_rate': 0.16222612210863313,
 'policy_cost_per_capita': 0.9998800675751829,
 'policy_type': 'binary_gap',
 'd': 2}

In [8]:
# Can try a different budget without redoing the fit step and precomputation step.
# Note that budget must lie in the set of budgets used in the precomputation step.
# Setting the budget will clear assignments attribute.
tt.set_budget(1.5)
tt.run_opt(test_covariate_dataset)
res = tt.evaluate(test_dataset)
res


{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.008028498371086004,
 'post_transfer_poverty_rate': 0.02303141832593607,
 'policy_cost_per_capita': 1.4999873804655806,
 'policy_type': 'binary_gap',
 'd': 2}